# 07 — Hiérarchie des compétences : hard vs soft

On va répondre ici au constat de la classification hard/soft (924 hard / 1 soft), non par erreur, mais parce qu'ESCO ne relie presque jamais ses professions IT aux compétences transversales (soft), celles-ci existent dans une collection transversale séparée.


In [ ]:
import os

import csv, re
import jobkb_common as C
from collections import Counter, defaultdict

In [2]:
def load(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))
def split_multi(cell):
    return [x.strip() for x in (cell or "").split("\n") if x.strip()]

## 7.1. Couche *soft* : la collection transversale d'ESCO

La collection transversale d'ESCO est une liste curatée de compétences transversales (soft) : « travailler en équipe », « penser de manière critique », « résoudre les conflits », « gérer le temps »… On l'ingère comme compétences soft. Les entrées déjà présentes (liées à une profession IT) sont reclassées hard→soft, les autres sont ajoutées.

In [3]:
transv = C.read_csv_smart(os.path.join(C.ESCO_DIR, "transversalSkillsCollection_fr.csv"))
existing_skills = load(C.SKILLS_CSV)

soft_rows, soft_label_rows = [], []
for _, t in transv.iterrows():
    sid = C.uri_tail(t["conceptUri"])
    eid = C.mint_id("SKL_", "ESCO", sid)
    alts = split_multi(t["altLabels"])
    soft_rows.append({
        "entity_id": eid, "source": "ESCO", "source_id": sid,
        "pref_label_fr": t["preferredLabel"], "pref_label_en": "",
        "alt_labels_fr": " | ".join(alts), "alt_labels_en": "",
        "description_fr": (t["description"] or "").replace("\n", " "), "description_en": "",
        "esco_skill_type": t["skillType"], "esco_reuse_level": t["reuseLevel"],
        "hard_soft_provisional": "soft", "hard_soft_method": "esco_transversal_collection",
    })
    soft_label_rows += C.make_label_rows(eid, "skill", "ESCO",
        preferred={"fr": [t["preferredLabel"]]}, alts={"fr": alts})

transv_eids = {r["entity_id"] for r in soft_rows}
n_reclassified = sum(1 for s in existing_skills if s["entity_id"] in transv_eids)
print(f"Collection transversale : {len(soft_rows)} compétences soft")
print(f"  dont déjà présentes (reclassées hard→soft) : {n_reclassified}")
print(f"  nouvelles compétences soft ajoutées        : {len(soft_rows) - n_reclassified}")
print("\nExemples de soft skills :")
for r in soft_rows[:8]:
    print("   -", r["pref_label_fr"])

Collection transversale : 95 compétences soft
  dont déjà présentes (reclassées hard→soft) : 1
  nouvelles compétences soft ajoutées        : 94

Exemples de soft skills :
   - faire preuve d’initiative
   - adopter des stratégies de promotion de la biodiversité et du bien-être animal
   - conseiller d’autres personnes
   - accepter les critiques et les directives
   - mettre en application des connaissances dans les domaines de la philosophie, de l’éthique et de la religion
   - adopter des stratégies de réduction de la pollution
   - respecter ses engagements
   - s’adapter aux exigences physiques


## 7.2. Hiérarchie des compétences

ESCO organise ses compétences sous des **groupes de compétences** (`SkillGroup`) via le fichier `broaderRelationsSkillPillar_fr`. On crée un nœud par groupe utile et une arête `compétence → groupe` (`broader_than`). Ces groupes fournissent une taxonomie structurelle propre.

In [4]:
broader = C.read_csv_smart(os.path.join(C.ESCO_DIR, "broaderRelationsSkillPillar_fr.csv"))

# on part de l'ensemble fusionné (hard existants + soft ajoutés)
# exclure les nœuds de groupes d'un run précédent (dérivés) pour rester idempotent

base_skills = [s for s in existing_skills if s.get("esco_skill_type") != "skill_group"]
merged = {(s["source"], s["source_id"]): s for s in base_skills}
for r in soft_rows:
    merged[(r["source"], r["source_id"])] = r
skill_eid_by_sid = {s["source_id"]: s["entity_id"] for s in merged.values() if s["source"] == "ESCO"}

group_nodes, group_label = {}, {}
skill_hier_rows = []
for _, r in broader.iterrows():
    child = C.uri_tail(r["conceptUri"])
    if child not in skill_eid_by_sid or r["broaderType"] != "SkillGroup":
        continue
    gsid = C.uri_tail(r["broaderUri"])
    if gsid not in group_nodes:
        group_nodes[gsid] = C.mint_id("SKL_", "ESCO", gsid)
        group_label[gsid] = r["broaderLabel"]
    skill_hier_rows.append({
        "parent_entity_id": group_nodes[gsid],
        "child_entity_id": skill_eid_by_sid[child],
        "entity_kind": "skill", "relation_type": "broader_than", "source": "ESCO",
    })

# créer les nœuds de groupes (pour que les extrémités d'arêtes existent)
group_skill_rows, group_label_rows = [], []
for gsid, geid in group_nodes.items():
    group_skill_rows.append({
        "entity_id": geid, "source": "ESCO", "source_id": gsid,
        "pref_label_fr": group_label[gsid], "pref_label_en": group_label[gsid],
        "alt_labels_fr": "", "alt_labels_en": "",
        "description_fr": "Groupe de compétences ESCO", "description_en": "",
        "esco_skill_type": "skill_group", "esco_reuse_level": "",
        "hard_soft_provisional": "group", "hard_soft_method": "esco_skill_group",
    })
    group_label_rows += C.make_label_rows(geid, "skill", "ESCO", preferred={"fr": [group_label[gsid]]})

print(f"{len(group_nodes)} groupes de compétences, {len(skill_hier_rows)} arêtes compétence→groupe")

186 groupes de compétences, 1019 arêtes compétence→groupe


## 7.3. Sous-typage des compétences *hard* propres à l'informatique

Pour distinguer les natures de hard skills (langage comme Python, socle technique comme Kubernetes, méthode comme *graph learning*), on va s'appuier d'abord sur le **groupe ESCO** de la compétence, complété par un repérage lexical des langages/technologies nommés.

In [5]:
geid_to_label = {geid: group_label[gsid] for gsid, geid in group_nodes.items()}
group_of_skill = defaultdict(list)
for h in skill_hier_rows:
    group_of_skill[h["child_entity_id"]].append(h["parent_entity_id"])

TECH_RE = re.compile(r"\b(python|java|javascript|typescript|c\+\+|c#|php|ruby|go|golang|rust|"
                     r"kotlin|swift|scala|sql|html|css|kubernetes|docker|tensorflow|pytorch)\b")

def subtype_hard(skill):
    lab = skill["pref_label_fr"].lower()
    if TECH_RE.search(lab):
        return "langage_ou_techno_nommee"
    for geid in group_of_skill.get(skill["entity_id"], []):
        gl = geid_to_label.get(geid, "").lower()
        if "programming" in gl:            return "programmation"
        if "database" in gl or "data" in gl: return "donnees_bdd"
        if "network" in gl:                return "reseau"
        if "security" in gl or "protecting" in gl or "penetration" in gl: return "securite"
        if "development" in gl or "design" in gl: return "developpement_conception"
    return "autre_hard"

# annoter chaque compétence hard avec son sous-type dans hard_soft_method (suffixe)
for key, s in merged.items():
    if s["hard_soft_provisional"] == "hard":
        s["it_subtype"] = subtype_hard(s)
    elif s["hard_soft_provisional"] == "soft":
        s["it_subtype"] = "soft_transversale"
    else:
        s["it_subtype"] = "groupe"

hard = [s for s in merged.values() if s["hard_soft_provisional"] == "hard"]
print(f"Sous-typage IT des {len(hard)} compétences hard :")
for k, v in Counter(s["it_subtype"] for s in hard).most_common():
    print(f"   {k}: {v}")

Sous-typage IT des 3146 compétences hard :
   autre_hard: 2718
   developpement_conception: 188
   donnees_bdd: 150
   programmation: 36
   langage_ou_techno_nommee: 27
   securite: 20
   reseau: 7


## 7.4. Écriture en format canonique

On ajoute le champ `it_subtype` au schéma des compétences (colonne supplémentaire), on écrit l'ensemble fusionné (hard + soft + groupes) et les arêtes de hiérarchie.

In [6]:
# schéma étendu avec it_subtype
SKILL_FIELDS_EXT = C.SKILL_FIELDS + ["it_subtype"]

all_skill_rows = list(merged.values()) + group_skill_rows
for r in all_skill_rows:
    r.setdefault("it_subtype", "groupe" if r["hard_soft_provisional"] == "group" else "autre_hard")

# remplacement ESCO ; ROME/Wikidata skills éventuels préservés
def write_skills(rows):
    C.ensure_dirs()
    with open(C.SKILLS_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=SKILL_FIELDS_EXT)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in SKILL_FIELDS_EXT})

# non-ESCO éventuels, hors nœuds de groupes dérivés
non_esco = [s for s in load(C.SKILLS_CSV)
            if s["source"] != "ESCO" and s.get("esco_skill_type") != "skill_group"]
for s in non_esco:
    s.setdefault("it_subtype", "autre_hard")
write_skills(all_skill_rows + non_esco)

# labels des soft + groupes (remplacement par source)
existing_labels = [l for l in load(C.LABELS_CSV)]
# on ajoute les nouveaux labels soft + groupes s'ils n'existent pas déjà
have = {(l["entity_id"], l["label_norm"], l["label_type"]) for l in existing_labels}
new_labels = [l for l in (soft_label_rows + group_label_rows)
              if (l["entity_id"], l["label_norm"], l["label_type"]) not in have]
C.ensure_dirs()
with open(C.LABELS_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=C.LABEL_FIELDS)
    w.writeheader()
    for l in existing_labels + new_labels:
        w.writerow({k: l.get(k, "") for k in C.LABEL_FIELDS})

# hiérarchie des compétences
existing_hier = [h for h in load(C.HIERARCHY_CSV)]
kept_hier = [h for h in existing_hier if not (h["source"] == "ESCO" and h["entity_kind"] == "skill")]
with open(C.HIERARCHY_CSV, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=C.HIERARCHY_FIELDS)
    w.writeheader()
    for h in kept_hier + skill_hier_rows:
        w.writerow({k: h.get(k, "") for k in C.HIERARCHY_FIELDS})

C.log_provenance("ESCO_SKILLS_HIER", [{
    "entity_id": "ALL_SKILLS_HIER", "source": "ESCO_SKILLS_HIER",
    "source_version": "transversal+broader (fr)", "retrieved_at": C.now_iso(),
    "retrieval_method": "transversalSkillsCollection + broaderRelationsSkillPillar",
    "notes": f"{len(soft_rows)} soft, {len(group_nodes)} groupes, {len(skill_hier_rows)} arêtes",
}])
print("Écrit : skills.csv (+it_subtype), labels.csv, hierarchy.csv")

Écrit : skills.csv (+it_subtype), labels.csv, hierarchy.csv


In [10]:
# Synthèse
skills = load(C.SKILLS_CSV)
print("=== Bilan hard/soft===")
print("hard/soft/group :", dict(Counter(s["hard_soft_provisional"] for s in skills)))
print("\nSous-types IT (hard) :")
for k, v in Counter(s["it_subtype"] for s in skills if s["hard_soft_provisional"] == "hard").most_common():
    print(f"   {k}: {v}")

# contrôle d'intégrité de la hiérarchie des compétences
hier = [h for h in load(C.HIERARCHY_CSV) if h["entity_kind"] == "skill"]
sids = {s["entity_id"] for s in skills}
dangling = [h for h in hier if h["parent_entity_id"] not in sids or h["child_entity_id"] not in sids]
assert not dangling, f"{len(dangling)} arêtes de compétences pendantes !"
print(f"\nHiérarchie compétences : {len(hier)} arêtes, intégrité OK")

# exemple de branche soft + hard
print("\nExemple soft :", next(s['pref_label_fr'] for s in skills if s['hard_soft_provisional']=='soft'))
print("Exemple hard :",
      next((s['pref_label_fr'] for s in skills if s.get('it_subtype')=='programmation'), '—'))

=== Bilan hard/soft===
hard/soft/group : {'hard': 5369, 'soft': 122, 'group': 186}

Sous-types IT (hard) :
   autre_hard: 4941
   developpement_conception: 188
   donnees_bdd: 150
   programmation: 36
   langage_ou_techno_nommee: 27
   securite: 20
   reseau: 7

Hiérarchie compétences : 1019 arêtes, intégrité OK

Exemple soft : Faire preuve d'autonomie
Exemple hard : analyser les spécifications du logiciel
